# Installation/Setup
- for Colab running

In [ ]:
# @title Install DSSP, ClustalO, MUSCLE
! apt-get -qq update && apt-get -qq install -y dssp clustalo
! wget -q https://github.com/rcedgar/muscle/releases/download/v5.3/muscle-linux-x86.v5.3 -O /content/muscle
! chmod +x /content/muscle

# @title Clone BE3D (pipeline package + example scripts + example data)
# @markdown Branch used below is a working branch, not main -- repoint at main once merged.
BECLUST3D_BRANCH = "feature/optimize-colab-notebooks"
! rm -rf /content/beclust3d-public
! git clone --quiet --branch {BECLUST3D_BRANCH} --depth 1 https://github.com/broadinstitute/BE3D.git /content/beclust3d-public

# @title Install Python dependencies
# be3d_local.py runs as its own subprocess (sys.executable) importing the full beclust3d
# package, not just the plotting helpers imported directly below -- installing the
# cloned package itself (rather than hand-listing packages here) pulls in everything
# pyproject.toml declares (numpy, biopython, biopandas, DSSPparser, wget, etc.), so this
# stays in sync with the package's real dependencies instead of drifting from them.
! pip install -q /content/beclust3d-public
! pip install -q ipymolstar

import os
import sys
import subprocess
import copy
import yaml
import pandas as pd
import numpy as np
import plotly.io as pio
from IPython.display import display, Image, SVG
from ipywidgets import interact, Dropdown
from ipymolstar import PDBeMolstar

# Colab's own Plotly renderer -- see BE3D_local.ipynb's Setup cell for why this matters:
# leaving pio.renderers.default on whatever got auto-detected can resolve to MULTIPLE
# renderers at once (e.g. "colab+notebook_connected"), so every fig.show() call renders
# once per registered renderer, stacking visible duplicates of every single plot.
pio.renderers.default = 'colab'

# Colab's 'colab' renderer loads its JS bundle asynchronously the FIRST time a figure is
# shown in a given session. A display()/show() call that fires before that finishes loading
# renders a permanently blank output (it doesn't retry) -- later figures then render fine
# once the bundle is cached. Warming it up here with a throwaway figure, before any real
# plot runs, avoids ever hitting that race on a plot that actually matters.
import plotly.graph_objects as _go
display(_go.Figure())

BECLUST3D_PATH = '/content/beclust3d-public'
sys.path.insert(0, BECLUST3D_PATH)
sys.path.insert(0, os.path.join(BECLUST3D_PATH, 'examples'))

from be3d_local_helper import (
    show_svgs, show_images, plot_residue_dot, plot_ppi_vs_noppi_scatter,
    render_molstar, load_molstar_pdb, color_molstar, chain_values_from_df,
)
from be3d_plotly import (
    show_side_by_side, show_stacked, show_picker, plot_hypothesis_qa, plot_violin_by_processed_muttype, plot_score_scatter,
    plot_dendrogram, plot_meta_dendrogram, plot_lfc_lfc3d_scatter, plot_plddt_rsa_scatter,
    plot_domain_barplot, plot_plddt_dis_barplot, plot_enrichment_test,
    plot_meta_score_scatter, plot_meta_lfc_lfc3d_scatter, plot_meta_plddt_rsa_scatter,
    plot_meta_domain_barplot, plot_meta_plddt_dis_barplot,
    COLOR_POS, COLOR_NEG,
)

def run_be3d(yaml_path):
    script = os.path.join(BECLUST3D_PATH, 'examples', 'be3d_local.py')
    # Capture + print explicitly rather than letting the child inherit stdout/stderr --
    # a subprocess's inherited file descriptors don't reliably show up in a notebook
    # cell's own output (Jupyter/Colab capture sys.stdout at the Python level, which a
    # child process's raw fd can bypass), so check=True alone can raise CalledProcessError
    # with no visible clue about what actually went wrong inside be3d_local.py.
    result = subprocess.run([sys.executable, script, yaml_path], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        result.check_returncode()

def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

def run_be3d_if_needed(yaml_path, done_marker):
    if os.path.exists(done_marker):
        print(f'[skip] {done_marker} already exists')
    else:
        run_be3d(yaml_path)


# Settings
- Check user intent: monomer, ppi mode, blind-target mode
- Show default settings
- As an example, we can use KBTBD4–HDAC1 case which supports monomer, ppi and ppi(blind) mode examples


In [ ]:
# KBTBD4-HDAC1 (8VOJ) supports all three modes: monomer, ppi (via ppi_diff), and blind_target.
# These yaml configs (and the example screens/PDB they point at) were cloned above into
# /content/beclust3d-public/examples/{yaml,data,pdb} -- pipeline outputs go to a separate
# /content/BE3D_example/output/ scratch dir, not into the cloned repo tree.
YAML_DIR = '/content/beclust3d-public/examples/yaml'
MONOMER_YAML = f'{YAML_DIR}/KBTBD4_chain_B_colab.yaml'
PPI_YAML = f'{YAML_DIR}/ppi_diff_KBTBD4_HDAC1_colab.yaml'
BLIND_TARGET_YAML = f'{YAML_DIR}/blind_target_KBTBD4_HDAC1_colab.yaml'

for label, path in [('monomer', MONOMER_YAML), ('ppi (ppi_diff)', PPI_YAML), ('blind_target', BLIND_TARGET_YAML)]:
    cfg = load_yaml(path)
    print(f"--- {label}: mode='{cfg.get('mode')}' ---")
    shown = {k: cfg[k] for k in ('input_gene', 'input_uniprot', 'input_chain', 'output_dir') if k in cfg}
    print(yaml.safe_dump(shown, sort_keys=False))


# Monomer Mode

## BE-QA
- QA plots (KS2, MW)
- QA plot (violin plot)

In [ ]:
monomer_cfg = load_yaml(MONOMER_YAML)
monomer_dir, monomer_gene, monomer_uniprot = monomer_cfg['output_dir'], monomer_cfg['input_gene'], monomer_cfg['input_uniprot']
run_be3d_if_needed(MONOMER_YAML, os.path.join(monomer_dir, 'RUN_COMPLETED.txt'))

monomer_screens = [s.strip().split('.')[0] for s in monomer_cfg['screens'].split(',')]

monomer_hyp_ks = plot_hypothesis_qa(monomer_dir, test='KolmogorovSmirnov')
monomer_hyp_mw = plot_hypothesis_qa(monomer_dir, test='MannWhitney')

def show_monomer_qa(screen_name):
    print('QA (KS2, MW test, all screens) and processed LFC distribution by mutation category '
          '(violin, post mutation_priority + per-category filtering):')
    violin_fig = plot_violin_by_processed_muttype(monomer_dir, monomer_gene, screen_name)
    show_side_by_side(monomer_hyp_ks, monomer_hyp_mw, violin_fig, width=600, height=400, spacing=0.08)

interact(show_monomer_qa, screen_name=Dropdown(options=monomer_screens, description='Screen:'));


## BE-Clust3D
- Residue dot-plot for single screens (LFC and LFC3D)
- Scatter plots of LFC and LFC3D comparison for single screens
- Dendrogram of LFC and LFC3D

In [ ]:
def show_monomer_clust3d(screen_name):
    print('Residue dot-plots, LFC (positive, negative):')
    show_side_by_side(
        plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='positive'),
        plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='negative'),
		width=600, height=400
    )
    print('Residue dot-plots, LFC3D (positive, negative):')
    show_side_by_side(
        plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='positive'),
        plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='negative'),
		width=600, height=400
    )

    print('LFC vs. LFC3D (residues with LFC3D but no LFC shown in the left strip):')
    fig = plot_lfc_lfc3d_scatter(monomer_dir, monomer_gene, screen_name, width=500, height=400)
    if fig is not None:
        display(fig)

    print('pLDDT vs. RSA / LFC3D hit count by domain / pLDDT-disorder category / Enrichment test (pLDDT-disorder, log2 odds ratio):')
    show_side_by_side(
        plot_plddt_rsa_scatter(monomer_dir, monomer_gene, screen_name),
        plot_domain_barplot(monomer_dir, monomer_gene, monomer_uniprot, screen_name),
        plot_plddt_dis_barplot(monomer_dir, monomer_gene, screen_name),
        plot_enrichment_test(monomer_dir, monomer_gene, screen_name=screen_name),
        # , spacing=0.5
        width=600
    )

    print('Dendrogram (p<0.05) -- pick a score type / direction:')
    show_picker({
        'LFC positive': plot_dendrogram(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='Positive', height=400),
        'LFC negative': plot_dendrogram(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='Negative', height=400),
        'LFC3D positive': plot_dendrogram(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='Positive', height=400),
        'LFC3D negative': plot_dendrogram(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='Negative', height=400),
    }, description='Dendrogram:')

interact(show_monomer_clust3d, screen_name=Dropdown(options=monomer_screens, description='Screen:'));


## BE-MetaClust3D
- If multiple screens
- Residue dot-plot for single screens (meta-LFC and meta-LFC3D)
- Scatter plots of meta-LFC and meta-LFC3D comparison for single screens
- Dendrogram of meta-LFC and -metaLFC3D

In [ ]:
monomer_func_meta = monomer_cfg['function_for_meta']

if len(monomer_screens) > 1:
    print('Meta residue dot-plots, meta-LFC (positive, negative):')
    show_side_by_side(
        plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='positive'),
        plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='negative'),
		width=600, height=400
    )
    print('Meta residue dot-plots, meta-LFC3D (positive, negative):')
    show_side_by_side(
        plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='positive'),
        plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='negative'),
		width=600, height=400
    )

    print('meta-LFC vs. meta-LFC3D (residues with meta-LFC3D but no meta-LFC shown in the left strip):')
    fig = plot_meta_lfc_lfc3d_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, width=500, height=400)
    if fig is not None:
        display(fig)

    print('pLDDT vs. RSA / Meta LFC3D hit count by domain / pLDDT-disorder category / Meta enrichment test (pLDDT-disorder, log2 odds ratio):')
    show_side_by_side(
        plot_meta_plddt_rsa_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta),
        plot_meta_domain_barplot(monomer_dir, monomer_gene, monomer_uniprot, function_for_meta=monomer_func_meta),
        plot_meta_plddt_dis_barplot(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta),
        plot_enrichment_test(monomer_dir, monomer_gene, screen_name=None),
        width=600
    )

    print('Meta dendrogram (p<0.05) -- pick a score type / direction:')
    show_picker({
        'Meta LFC positive': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='Positive', height=400),
        'Meta LFC negative': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='Negative', height=400),
        'Meta LFC3D positive': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='Positive', height=400),
        'Meta LFC3D negative': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='Negative', height=400),
    }, description='Dendrogram:')
else:
    print('Only one screen -- no meta-aggregation to show.')


# PPI mode

## BE-Clust3D
- Residue dot-plot for single screens (LFC and LFC3D) and PPI mode LFC3D
- Scatter plots of single screen LFC3Ds (x-axis) vs PPI mode LFC3D (y-axis)
- Dendrogram of single screen  LFC and LFC3D and PPI mode LFC3D

In [ ]:
ppi_cfg = load_yaml(PPI_YAML)
ppi_root = ppi_cfg['output_dir']
gene_names = [g.strip() for g in ppi_cfg['input_gene'].split(',')]
chain_list = [c.strip() for c in ppi_cfg['input_chain'].split(',')]
ppi_screens = [s.strip().split('.')[0] for s in ppi_cfg['screens'].split(',')]

# mode: ppi_diff runs the PPI leg (mode: complex) and no-PPI leg (mode: monomer, per gene)
# once, then merges -- skip_existing makes each pass a no-op for the pipeline legs once the
# first pass has run them. score_type controls only the (cheap) merge step: 'LFC3D' produces
# one merged TSV+PDB set per screen; 'Meta_LFC3D' produces the meta-aggregated one (needed by
# the BE-MetaClust3D and Merged-results sections below).
def run_ppi_diff_pass(score_type):
    variant_cfg = copy.deepcopy(ppi_cfg)
    variant_cfg['score_type'] = score_type
    variant_yaml = os.path.join('/content/BE3D_example', f'_ppi_diff_{score_type}.yaml')
    os.makedirs('/content/BE3D_example', exist_ok=True)
    with open(variant_yaml, 'w') as f:
        yaml.safe_dump(variant_cfg, f)
    run_be3d(variant_yaml)

run_ppi_diff_pass('LFC3D')
run_ppi_diff_pass('Meta_LFC3D')

ppi_func_meta = ppi_cfg['function_for_meta']
ppi_uniprot_by_gene = dict(zip(gene_names, [u.strip() for u in ppi_cfg['input_uniprot'].split(',')]))

def show_ppi_clust3d(gene, screen_name):
    chain = dict(zip(gene_names, chain_list))[gene]
    uniprot = ppi_uniprot_by_gene[gene]
    noppi_dir = os.path.join(ppi_root, 'no_ppi', gene)
    ppi_dir = os.path.join(ppi_root, 'ppi', gene)

    print(f'{gene} (chain {chain}) -- residue dot-plots, LFC3D positive -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_score_scatter(noppi_dir, gene, screen_name, score_type='LFC3D', direction='positive'),
        plot_score_scatter(ppi_dir, gene, screen_name, score_type='LFC3D', direction='positive'),
    )
    print('residue dot-plots, LFC3D negative -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_score_scatter(noppi_dir, gene, screen_name, score_type='LFC3D', direction='negative'),
        plot_score_scatter(ppi_dir, gene, screen_name, score_type='LFC3D', direction='negative'),
    )

    print('no-PPI LFC3D (x) vs. PPI-mode LFC3D (y):')
    df_screen = pd.read_csv(os.path.join(ppi_root, f'ppi_vs_noppi_{screen_name}.tsv'), sep='\t')
    plot_ppi_vs_noppi_scatter(df_screen[df_screen['gene'] == gene], 'LFC3D')

    print('LFC vs. LFC3D -- no-PPI, then PPI-mode (residues with LFC3D but no LFC shown in each left strip):')
    show_side_by_side(
        plot_lfc_lfc3d_scatter(noppi_dir, gene, screen_name),
        plot_lfc_lfc3d_scatter(ppi_dir, gene, screen_name),
    )

    print('pLDDT vs. RSA -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_plddt_rsa_scatter(noppi_dir, gene, screen_name),
        plot_plddt_rsa_scatter(ppi_dir, gene, screen_name),
    )

    print('LFC3D hit count by pLDDT-disorder category -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_plddt_dis_barplot(noppi_dir, gene, screen_name),
        plot_plddt_dis_barplot(ppi_dir, gene, screen_name),
    )

    print('Enrichment test (pLDDT-disorder) -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_enrichment_test(noppi_dir, gene, screen_name=screen_name),
        plot_enrichment_test(ppi_dir, gene, screen_name=screen_name),
    )

    print('LFC3D dendrogram (p<0.05) -- pick a mode / direction:')
    show_picker({
        'no-PPI positive': plot_dendrogram(noppi_dir, gene, screen_name, score_type='LFC3D', direction='Positive'),
        'PPI-mode positive': plot_dendrogram(ppi_dir, gene, screen_name, score_type='LFC3D', direction='Positive'),
        'no-PPI negative': plot_dendrogram(noppi_dir, gene, screen_name, score_type='LFC3D', direction='Negative'),
        'PPI-mode negative': plot_dendrogram(ppi_dir, gene, screen_name, score_type='LFC3D', direction='Negative'),
    }, description='Dendrogram:')

interact(
    show_ppi_clust3d,
    gene=Dropdown(options=gene_names, description='Gene:'),
    screen_name=Dropdown(options=ppi_screens, description='Screen:'),
);


## BE-MetaClust3D
- If multiple screens
- Residue dot-plot for single screens (meta-LFC and meta-LFC3D) and PPI mode meta-LFC3D
- Scatter plots of meta-LFC3D (single screens, x-axis) vs meta-LFC3D (PPI mode, y-axis)
- Dendrogram of meta-LFC and meta-LFC3D (single screens) and meta-LFC3D (PPI-mode)

In [ ]:
df_merged = pd.read_csv(os.path.join(ppi_root, 'ppi_vs_noppi_Meta_LFC3D.tsv'), sep='\t')
df_merged_sorted = df_merged.reindex(df_merged['delta_score'].abs().sort_values(ascending=False).index)

print('Top 10 residues by |delta meta-LFC3D| (PPI - no-PPI):')
top10 = df_merged_sorted.head(10)
display(top10[['gene', 'chain', 'unipos', 'unires', 'noppi_score', 'ppi_score', 'delta_score']])
top10_unipos = top10['unipos'].tolist()

base_pdb = os.path.join(ppi_root, 'ppi', gene_names[0], 'sequence_structure')
base_pdb = os.path.join(base_pdb, [f for f in os.listdir(base_pdb) if f.endswith('_processed.pdb')][0])

merged_views = {
    'No-PPI meta-LFC3D': chain_values_from_df(df_merged, 'noppi_score'),
    'PPI-mode meta-LFC3D': chain_values_from_df(df_merged, 'ppi_score'),
    'Delta (PPI - no-PPI) meta-LFC3D': chain_values_from_df(df_merged, 'delta_score'),
}

print('Structure colored by the selected view (spheres = the top 10 |delta| residues above):')
merged_widget = PDBeMolstar(hide_water=True, height='333px')
load_molstar_pdb(merged_widget, base_pdb)
display(merged_widget)

def show_merged_view(view_name):
    color_molstar(merged_widget, merged_views[view_name], vmax=2.0, highlight_top_n=10)

interact(show_merged_view, view_name=Dropdown(options=list(merged_views), description='View:'));


# Blind target mode
- in a table, highlight those have LFC3D values (single and meta)
- Residue-dot plot for the above table (one screen, then use than, if multiple screens, then use meta-LFC3D)
- Use Molstar viewer to visualize the new LFC3D value in 3D structure. Blue for + and Neg for - LFC3D  (one screen, then use than, if multiple screens, then use meta-LFC3D)

In [ ]:
blind_cfg = load_yaml(BLIND_TARGET_YAML)
blind_dir = blind_cfg['output_dir']
blind_gene, blind_chain = blind_cfg['input_gene'], blind_cfg['input_chain']
blind_tsv = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D.tsv')

if not os.path.exists(blind_tsv):
    run_be3d(BLIND_TARGET_YAML)

df_blind = pd.read_csv(blind_tsv, sep='\t')

# use the meta-aggregated column if there's more than one partner screen, else the single screen's
screen_names = [c[:-len('_LFC3D_blind_overall')] for c in df_blind.columns if c.endswith('_LFC3D_blind_overall')]
if 'Meta_LFC3D_blind_overall' in df_blind.columns:
    neg_col, pos_col, overall_col = 'Meta_LFC3D_blind_neg', 'Meta_LFC3D_blind_pos', 'Meta_LFC3D_blind_overall'
else:
    screen_name = screen_names[0]
    neg_col, pos_col, overall_col = f'{screen_name}_LFC3D_blind_neg', f'{screen_name}_LFC3D_blind_pos', f'{screen_name}_LFC3D_blind_overall'

df_blind_hits = df_blind[(df_blind[neg_col] != '-') | (df_blind[pos_col] != '-')]
print(f'{blind_gene} (chain {blind_chain}) -- {len(df_blind_hits)}/{len(df_blind)} residues have a blind LFC3D value:')
display(df_blind_hits[['unipos', 'unires', 'chain', neg_col, pos_col, overall_col]])

print('Residue dot-plot (signed value: negative or positive column, whichever is set):')
signed = pd.to_numeric(df_blind[neg_col].replace('-', pd.NA), errors='coerce')
signed = signed.fillna(pd.to_numeric(df_blind[pos_col].replace('-', pd.NA), errors='coerce'))
df_blind_signed = df_blind.copy()
df_blind_signed['_signed_blind_LFC3D'] = signed
plot_residue_dot(df_blind_signed, '_signed_blind_LFC3D', f'{blind_gene} blind LFC3D')

print('3D structure colored by the selected view (blue = positive, red = negative):')
overall_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_overall.pdb')
pos_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_pos.pdb')
neg_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_neg.pdb')
# run_blind_target always writes all three PDBs together (same coordinates, different
# B-factors baked in) when user_pdb is set, so which one is loaded as the base structure
# doesn't matter -- only the color_data changes per dropdown selection below.
blind_base_pdb = overall_pdb if os.path.exists(overall_pdb) else (pos_pdb if os.path.exists(pos_pdb) else neg_pdb)

def _blind_chain_values(col):
    values = pd.to_numeric(df_blind[col].replace('-', pd.NA), errors='coerce')
    return {blind_chain: {int(p): float(v) for p, v in zip(df_blind['unipos'], values) if pd.notna(v)}}

blind_views = {
    'Negative': _blind_chain_values(neg_col),
    'Positive': _blind_chain_values(pos_col),
    'Overall': _blind_chain_values(overall_col),
}

blind_widget = PDBeMolstar(hide_water=True, height='333px')
load_molstar_pdb(blind_widget, blind_base_pdb)
display(blind_widget)

def show_blind_view(view_name):
    color_molstar(blind_widget, blind_views[view_name], vmax=2.0, highlight_top_n=10)

interact(show_blind_view, view_name=Dropdown(options=list(blind_views), description='View:'));
